# 00 — New Data Exploration (Sept 2025 `charging_activity` + `evse_location`)

**Standalone exploration notebook — NOT part of the 01-05 pipeline.** Purpose: understand the
real structure of the two new restricted files before rewriting `01_data_loading.ipynb` (and
downstream `02`-`05`) to use them instead of `join_august2025.csv` / `OpenStreetEV_GLA.csv`.

Expected files (adjust filenames in the next cell once you've listed the folder):
- `03_data/restricted/evse_location.csv` — full EVSE location registry. Looks like an already-
  joined table across the Location > Device > EVSE > Connector hierarchy (lots of `_x`/`_y`
  suffixed columns), so expect multiple rows per physical on-street location.
- `03_data/restricted/charging_activity_sep.csv` — September 2025 charging sessions
  (`evse_uid`, `start_time`, `end_time`, `duration`).

Nothing here is a final deliverable — every print is a throwaway diagnostic. Once you've run
this, copy the **whole output of the last cell** (Section 4 summary) back into chat, plus
anything that looks unexpected along the way (especially the hierarchy counts, the coordinate
column names, and the `location_class` value counts).

In [1]:
import pandas as pd
import numpy as np
import os

BASE = "/Users/alexia/Documents/CASA/Dissertation"
RESTRICTED = os.path.join(BASE, "03_data/restricted")

print("Files in restricted folder:")
for f in sorted(os.listdir(RESTRICTED)):
    print(" -", f)


Files in restricted folder:
 - .DS_Store
 - charging_activity_sep.csv
 - evse_location.csv
 - gla
 - join_august2025.csv
 - winchester


## 1. Load `evse_location.csv` — check structure and hierarchy

In [2]:
loc_path = os.path.join(RESTRICTED, "evse_location.csv")  # rename if the listing above differs

# Try comma first; if it parses into a single column, the file is actually tab/other-separated
evse_location = pd.read_csv(loc_path)
if evse_location.shape[1] <= 1:
    print("Only 1 column parsed with comma -- retrying with tab separator")
    evse_location = pd.read_csv(loc_path, sep="\t")

print("Shape:", evse_location.shape)
print("\nColumns (copy this list back if it differs from what we discussed in chat):")
print(evse_location.columns.tolist())
print("\nDtypes:")
print(evse_location.dtypes)
evse_location.head(3)


Shape: (51386, 46)

Columns (copy this list back if it differs from what we discussed in chat):
['Source', 'id_x', 'address', 'city', 'postal_code', 'state', 'coordinates_latitude_x', 'coordinates_latitude_x.1', 'parking_type', 'operator_name', 'suboperator_name', 'owner_name', 'energy_mix', 'last_updated_x', 'created_at_x', 'location_class', 'location_id_x', 'zapmap_device_uid', 'power_band', 'device_max_power', 'location_id_y', 'evse_id', 'external_evse_uid', 'status_schedule', 'floor_level', 'coordinates_latitude_y', 'coordinates_longitude', 'physical_reference', 'directions', 'parking_restrictions', 'images', 'created_at_y', 'deleted_at_x', 'evse_uid', 'id_y', 'standard', 'format', 'power_type', 'max_voltage', 'max_amperage', 'max_electric_power', 'tariff_ids', 'terms_and_conditions', 'last_updated_y', 'created_at', 'deleted_at_y']

Dtypes:
Source                          str
id_x                            str
address                         str
city                            str

/var/folders/ns/mq6gshc5381576s51jjzgybm0000gn/T/ipykernel_95704/4288147575.py:4: DtypeWarning: Columns (0: external_evse_uid, 1: status_schedule, 2: directions, 3: parking_restrictions, 4: deleted_at_x) have mixed types. Specify dtype option on import or set low_memory=False.
  evse_location = pd.read_csv(loc_path)


,Source,id_x,address,city,postal_code,state,coordinates_latitude_x,coordinates_latitude_x.1,parking_type,operator_name,...,format,power_type,max_voltage,max_amperage,max_electric_power,tariff_ids,terms_and_conditions,last_updated_y,created_at,deleted_at_y
0,Custom,3SJEP0S,Dagenham,Dagenham,RM10 7XD,Barking and Dagenham,51.546391,51.546391,NaN,NaN,...,NaN,DC,NaN,NaN,50000.0,NaN,NaN,2025-05-19T12:42:31,2023-03-29T14:36:28,2025-05-19T00:00:00
1,Custom,3SJEP0S,Dagenham,Dagenham,RM10 7XD,Barking and Dagenham,51.546391,51.546391,NaN,NaN,...,NaN,DC,NaN,NaN,50000.0,NaN,NaN,2025-05-19T12:42:31,2023-03-29T14:36:28,2025-05-19T00:00:00
2,Custom,3SJEP0S,Dagenham,Dagenham,RM10 7XD,Barking and Dagenham,51.546391,51.546391,NaN,NaN,...,NaN,AC_3_PHASE,NaN,NaN,43000.0,NaN,NaN,2025-05-19T12:42:31,2023-03-29T14:36:28,2025-05-19T00:00:00


In [3]:
# Hierarchy check: Location > Device > EVSE > Connector should show STRICTLY INCREASING
# unique counts at each level (one Location can have multiple Devices, one Device multiple
# EVSEs/connectors). This is the same "count at the right level" issue that bit us with the
# old gla_evse_join.csv (device-level eⱼ was implausibly high, max 544).
hierarchy_cols_guess = {
    "Location id (id_x)":              "id_x",
    "Location id (location_id_x)":     "location_id_x",
    "Device id (zapmap_device_uid)":   "zapmap_device_uid",
    "EVSE id (evse_uid)":              "evse_uid",
    "EVSE id (evse_id)":               "evse_id",
    "Connector id (id_y)":             "id_y",
}

print("=== Unique counts per hierarchy level ===")
for label, col in hierarchy_cols_guess.items():
    if col in evse_location.columns:
        print(f"{label:35s} col='{col}':  {evse_location[col].nunique():>8,} unique  /  {evse_location.shape[0]:>8,} total rows")
    else:
        print(f"{label:35s} col='{col}':  NOT FOUND in columns")

print("\nExpectation: id_x (Location) should have the FEWEST unique values, id_y (Connector) the MOST.")
print("eⱼ (existing supply count in the demand formula) must be counted at the Location level, not below.")


=== Unique counts per hierarchy level ===
Location id (id_x)                  col='id_x':    23,013 unique  /    51,386 total rows
Location id (location_id_x)         col='location_id_x':    23,013 unique  /    51,386 total rows
Device id (zapmap_device_uid)       col='zapmap_device_uid':    32,697 unique  /    51,386 total rows
EVSE id (evse_uid)                  col='evse_uid':    38,358 unique  /    51,386 total rows
EVSE id (evse_id)                   col='evse_id':    38,358 unique  /    51,386 total rows
Connector id (id_y)                 col='id_y':    44,941 unique  /    51,386 total rows

Expectation: id_x (Location) should have the FEWEST unique values, id_y (Connector) the MOST.
eⱼ (existing supply count in the demand formula) must be counted at the Location level, not below.


In [4]:
# If both location_id_x and location_id_y (or id_x) exist, check they actually agree row-by-row
if "location_id_x" in evse_location.columns and "location_id_y" in evse_location.columns:
    mismatch = (evse_location["location_id_x"] != evse_location["location_id_y"]).sum()
    print(f"location_id_x != location_id_y in {mismatch:,} rows (should be 0, or explainable)")

if "id_x" in evse_location.columns and "location_id_x" in evse_location.columns:
    mismatch2 = (evse_location["id_x"] != evse_location["location_id_x"]).sum()
    print(f"id_x != location_id_x in {mismatch2:,} rows (should be 0 if both refer to the same Location id)")


location_id_x != location_id_y in 14 rows (should be 0, or explainable)
id_x != location_id_x in 0 rows (should be 0 if both refer to the same Location id)


In [5]:
# location_class -- equivalent of the old OpenStreetEV_GLA "location_category" field,
# used to filter down to On-street locations for eⱼ. Sample row showed "On-street" (lowercase
# street) vs the old data's "On-Street" -- do a case-insensitive check.
if "location_class" in evse_location.columns:
    print("=== location_class value counts (raw) ===")
    print(evse_location["location_class"].value_counts(dropna=False))
    print("\n=== value counts, case-insensitive ===")
    print(evse_location["location_class"].astype(str).str.lower().value_counts(dropna=False))
else:
    print("Column 'location_class' not found -- check the columns list above for the real name")


=== location_class value counts (raw) ===
location_class
On-street      36667
Destination    11442
En-route        1642
Other           1635
Name: count, dtype: int64

=== value counts, case-insensitive ===
location_class
on-street      36667
destination    11442
en-route        1642
other           1635
Name: count, dtype: int64


In [6]:
# Coordinate columns -- the pasted sample listed 'coordinates_latitude_x' TWICE, almost
# certainly a copy-paste slip (one of them should be a longitude field). Print every
# column with a coordinate-like name so we get the real field names from the actual file.
coord_cols = [c for c in evse_location.columns
              if any(k in c.lower() for k in ["coord", "lat", "lon", "lng"])]
print("Coordinate-like columns found:", coord_cols)
if coord_cols:
    print(evse_location[coord_cols].describe())


Coordinate-like columns found: ['coordinates_latitude_x', 'coordinates_latitude_x.1', 'coordinates_latitude_y', 'coordinates_longitude']
       coordinates_latitude_x  coordinates_latitude_x.1  \
count            51386.000000              51386.000000   
mean                51.504657                 51.504657   
std                  0.061119                  0.061119   
min                 51.299900                 51.299900   
25%                 51.466639                 51.466639   
50%                 51.509239                 51.509239   
75%                 51.544696                 51.544696   
max                 51.680623                 51.680623   

       coordinates_latitude_y  coordinates_longitude  
count            51386.000000           51386.000000  
mean                51.504657              -0.144330  
std                  0.061119               0.127827  
min                 51.299900              -0.491700  
25%                 51.466639              -0.222421  
5

In [7]:
# Missing value overview -- helps decide which columns are usable vs mostly-empty
# (e.g. tariff_ids, images, terms_and_conditions etc. are likely to be near-100% empty)
print("=== Missing value % per column (highest first) ===")
print((evse_location.isna().mean() * 100).round(1).sort_values(ascending=False).to_string())


=== Missing value % per column (highest first) ===
terms_and_conditions        100.0
deleted_at_x                100.0
floor_level                 100.0
status_schedule              95.8
parking_restrictions         95.0
directions                   84.2
parking_type                 77.9
deleted_at_y                 69.5
energy_mix                   67.2
suboperator_name             64.3
images                       64.1
tariff_ids                   53.1
owner_name                   36.7
operator_name                31.2
format                       25.6
max_amperage                 14.6
max_voltage                  14.3
external_evse_uid             3.6
physical_reference            3.2
max_electric_power            0.2
city                          0.2
power_type                    0.1
standard                      0.1
address                       0.0
last_updated_y                0.0
id_y                          0.0
evse_uid                      0.0
created_at                    0

In [8]:
# De-duplicate to LOCATION level -- this is the step that matters most for eⱼ.
# Fill in location_id_col / lat_col / lon_col once the cells above confirm the real names.
location_id_col = "id_x" if "id_x" in evse_location.columns else "location_id_x"

location_level = evse_location.drop_duplicates(subset=[location_id_col])
print(f"Rows in evse_location.csv:                {len(evse_location):,}")
print(f"Unique Locations ({location_id_col}):{'':>14}{len(location_level):,}")

if "location_class" in location_level.columns:
    is_onstreet = location_level["location_class"].astype(str).str.lower() == "on-street"
    print(f"Of which On-street (case-insensitive):     {is_onstreet.sum():,}")


Rows in evse_location.csv:                51,386
Unique Locations (id_x):              23,013
Of which On-street (case-insensitive):     21,367


## 2. Load `charging_activity_sep.csv` — check structure and date coverage

In [9]:
act_path = os.path.join(RESTRICTED, "charging_activity_sep.csv")  # rename if needed
charging_activity = pd.read_csv(act_path)
if charging_activity.shape[1] <= 1:
    print("Only 1 column parsed with comma -- retrying with tab separator")
    charging_activity = pd.read_csv(act_path, sep="\t")

print("Shape:", charging_activity.shape)
print("\nColumns:")
print(charging_activity.columns.tolist())
print("\nDtypes:")
print(charging_activity.dtypes)
charging_activity.head(3)


Shape: (524454, 4)

Columns:
['evse_uid', 'start_time', 'end_time', 'duration']

Dtypes:
evse_uid          str
start_time        str
end_time          str
duration      float64
dtype: object


,evse_uid,start_time,end_time,duration
0,0002729a6be20a87309637692e5a3af3,2025/9/7 22:21,2025/9/7 22:21,0.016667
1,0002729a6be20a87309637692e5a3af3,2025/9/7 22:21,2025/9/7 23:21,59.983333
2,0003081cf555fade0d74f080d8baa089,2025/9/11 21:49,2025/9/11 21:58,8.966667


In [10]:
# Parse datetimes and check the full-month coverage the supervisor mentioned
charging_activity["start_time"] = pd.to_datetime(charging_activity["start_time"])
charging_activity["end_time"]   = pd.to_datetime(charging_activity["end_time"])

print("Date range (start_time):", charging_activity["start_time"].min(), "to", charging_activity["start_time"].max())
print("Date range (end_time):  ", charging_activity["end_time"].min(), "to", charging_activity["end_time"].max())
print(f"\nTotal sessions: {len(charging_activity):,}")
print(f"Unique evse_uid: {charging_activity['evse_uid'].nunique():,}")


Date range (start_time): 2025-09-01 00:00:00 to 2025-09-30 23:59:00
Date range (end_time):   2025-09-01 00:01:00 to 2025-09-30 23:59:00

Total sessions: 524,454
Unique evse_uid: 18,748


In [11]:
# Validate duration == (end_time - start_time) in minutes -- if this holds, we can trust
# 'duration' directly instead of recomputing it, and it also confirms the timezone/rounding
# is consistent throughout the file.
computed_duration_min = (charging_activity["end_time"] - charging_activity["start_time"]).dt.total_seconds() / 60
diff = (charging_activity["duration"] - computed_duration_min).abs()
print("Max abs difference between given 'duration' and computed (end-start), in minutes:", diff.max())
print(f"Rows where difference > 0.01 min: {(diff > 0.01).sum():,} out of {len(charging_activity):,}")

print("\nDuration stats (minutes):")
print(charging_activity["duration"].describe())
print(f"\nNegative duration rows: {(charging_activity['duration'] < 0).sum():,}")
print(f"Zero duration rows: {(charging_activity['duration'] == 0).sum():,}")
print(f"Duration > 10,080 min (>7 days -- worth a look): {(charging_activity['duration'] > 10080).sum():,}")
print(f"Duration > 44,640 min (>31 days -- definitely suspicious): {(charging_activity['duration'] > 44640).sum():,}")


Max abs difference between given 'duration' and computed (end-start), in minutes: 0.9833333333333485
Rows where difference > 0.01 min: 498,885 out of 524,454

Duration stats (minutes):
count    524454.000000
mean         72.567316
std         128.268465
min           0.016667
25%           6.366667
50%          32.383333
75%          63.933333
max        1381.400000
Name: duration, dtype: float64

Negative duration rows: 0
Zero duration rows: 0
Duration > 10,080 min (>7 days -- worth a look): 0
Duration > 44,640 min (>31 days -- definitely suspicious): 0


In [12]:
# Same question we had to solve for join_august2025: exact duplicate or overlapping
# session rows for the same EVSE. If duplicates/overlaps exist, we'll need the same
# interval-merge + window-clip logic as 02_data_cleaning, just with a Sept 1 00:00 ->
# Oct 1 00:00 window instead of the old 7-day one.
dupe_exact = charging_activity.duplicated(subset=["evse_uid", "start_time", "end_time"]).sum()
print(f"Exact duplicate (evse_uid, start_time, end_time) rows: {dupe_exact:,}")

sessions_per_evse = charging_activity.groupby("evse_uid").size()
print("\nSessions per EVSE this month (distribution):")
print(sessions_per_evse.describe())


Exact duplicate (evse_uid, start_time, end_time) rows: 1,726

Sessions per EVSE this month (distribution):
count    18748.000000
mean        27.973864
std         78.703666
min          1.000000
25%          2.000000
50%          6.000000
75%         13.000000
max       1364.000000
dtype: float64


## 3. Cross-check: does `evse_uid` actually join the two files cleanly?

In [13]:
evse_uid_in_activity = set(charging_activity["evse_uid"].unique())
evse_uid_in_location = set(evse_location["evse_uid"].unique()) if "evse_uid" in evse_location.columns else set()

print(f"Unique evse_uid in charging_activity_sep.csv: {len(evse_uid_in_activity):,}")
print(f"Unique evse_uid in evse_location.csv:         {len(evse_uid_in_location):,}")
print(f"In both (joinable):                            {len(evse_uid_in_activity & evse_uid_in_location):,}")
print(f"In activity but NOT in location (orphan sessions -- should be ~0): {len(evse_uid_in_activity - evse_uid_in_location):,}")
print(f"In location but NOT in activity (zero-session EVSEs this month):   {len(evse_uid_in_location - evse_uid_in_activity):,}")


Unique evse_uid in charging_activity_sep.csv: 18,748
Unique evse_uid in evse_location.csv:         38,358
In both (joinable):                            18,748
In activity but NOT in location (orphan sessions -- should be ~0): 0
In location but NOT in activity (zero-session EVSEs this month):   19,610


## 4. Summary — copy this whole printed block back into chat

In [14]:
print("="*70)
print("SUMMARY TO COPY BACK")
print("="*70)
print(f"evse_location.csv: {evse_location.shape[0]:,} rows x {evse_location.shape[1]} cols")
print(f"  columns: {evse_location.columns.tolist()}")
print(f"charging_activity_sep.csv: {charging_activity.shape[0]:,} rows x {charging_activity.shape[1]} cols")
print(f"  date range: {charging_activity['start_time'].min()} to {charging_activity['start_time'].max()}")
print(f"  unique evse_uid: {charging_activity['evse_uid'].nunique():,}")
print(f"evse_uid join coverage: {len(evse_uid_in_activity & evse_uid_in_location):,} / {len(evse_uid_in_activity):,} activity-side EVSEs matched")
print()
print("(Please also paste back: the hierarchy unique-counts cell, the location_class value")
print(" counts, the coordinate column names, and the duration validation cell -- those five")
print(" pieces are what determine how we rewrite 01_data_loading.ipynb.)")


SUMMARY TO COPY BACK
evse_location.csv: 51,386 rows x 46 cols
  columns: ['Source', 'id_x', 'address', 'city', 'postal_code', 'state', 'coordinates_latitude_x', 'coordinates_latitude_x.1', 'parking_type', 'operator_name', 'suboperator_name', 'owner_name', 'energy_mix', 'last_updated_x', 'created_at_x', 'location_class', 'location_id_x', 'zapmap_device_uid', 'power_band', 'device_max_power', 'location_id_y', 'evse_id', 'external_evse_uid', 'status_schedule', 'floor_level', 'coordinates_latitude_y', 'coordinates_longitude', 'physical_reference', 'directions', 'parking_restrictions', 'images', 'created_at_y', 'deleted_at_x', 'evse_uid', 'id_y', 'standard', 'format', 'power_type', 'max_voltage', 'max_amperage', 'max_electric_power', 'tariff_ids', 'terms_and_conditions', 'last_updated_y', 'created_at', 'deleted_at_y']
charging_activity_sep.csv: 524,454 rows x 4 cols
  date range: 2025-09-01 00:00:00 to 2025-09-30 23:59:00
  unique evse_uid: 18,748
evse_uid join coverage: 18,748 / 18,748 act

In [15]:
# 检查9：同一个evse_uid内部是否存在时间段重叠
overlap_evse_count = 0
overlap_pair_count = 0
for uid, grp in charging_activity.groupby("evse_uid"):
    grp_sorted = grp.sort_values("start_time")
    starts = grp_sorted["start_time"].values[1:]
    prev_ends = grp_sorted["end_time"].values[:-1]
    overlaps = (starts < prev_ends).sum()
    if overlaps > 0:
        overlap_evse_count += 1
        overlap_pair_count += overlaps

print(f"存在内部重叠的evse_uid数量: {overlap_evse_count:,} / {charging_activity['evse_uid'].nunique():,}")
print(f"重叠的相邻session对总数: {overlap_pair_count:,}")

# 检查10：零session的EVSE是否集中在特定operator/Source
zero_session_uids = evse_uid_in_location - evse_uid_in_activity
zero_session_rows = evse_location[evse_location["evse_uid"].isin(zero_session_uids)].drop_duplicates(subset=["evse_uid"])
has_session_rows = evse_location[evse_location["evse_uid"].isin(evse_uid_in_activity)].drop_duplicates(subset=["evse_uid"])

print("=== Source分布对比：零session EVSE vs 有session EVSE ===")
print("\n零session EVSE的Source分布（占比%）：")
print((zero_session_rows["Source"].value_counts(normalize=True) * 100).round(1))
print("\n有session EVSE的Source分布（占比%）：")
print((has_session_rows["Source"].value_counts(normalize=True) * 100).round(1))

print("\n=== operator_name分布对比（缺失率也一并看）===")
print(f"零session EVSE中operator_name缺失率: {zero_session_rows['operator_name'].isna().mean()*100:.1f}%")
print(f"有session EVSE中operator_name缺失率: {has_session_rows['operator_name'].isna().mean()*100:.1f}%")

存在内部重叠的evse_uid数量: 427 / 18,748
重叠的相邻session对总数: 775
=== Source分布对比：零session EVSE vs 有session EVSE ===

零session EVSE的Source分布（占比%）：
Source
OCPI      72.8
Custom    27.2
Name: proportion, dtype: float64

有session EVSE的Source分布（占比%）：
Source
OCPI      90.9
Custom     9.1
Name: proportion, dtype: float64

=== operator_name分布对比（缺失率也一并看）===
零session EVSE中operator_name缺失率: 54.3%
有session EVSE中operator_name缺失率: 12.2%
